# Grouping books into simple categories

Run after `data-exploration.ipynb`. Map familiar categories first, then use a zero-shot classifier for the remaining books. The model downloads when it is first loaded.


In [ ]:
import pandas as pd
from tqdm.auto import tqdm
from transformers import pipeline

from book_recommender.analysis import predict_category

books = pd.read_csv("books_cleaned.csv", dtype={"isbn13": str})
books["categories"].value_counts().head(15)

## Map the known categories


In [ ]:
category_mapping = {
    "Fiction": "Fiction",
    "Juvenile Fiction": "Children's Fiction",
    "Biography & Autobiography": "Nonfiction",
    "History": "Nonfiction",
    "Literary Criticism": "Nonfiction",
    "Philosophy": "Nonfiction",
    "Religion": "Nonfiction",
    "Comics & Graphic Novels": "Fiction",
    "Drama": "Fiction",
    "Juvenile Nonfiction": "Children's Nonfiction",
    "Science": "Nonfiction",
    "Poetry": "Fiction",
}
books["simple_categories"] = books["categories"].map(category_mapping)
books["simple_categories"].value_counts(dropna=False)

## Load the classifier

Use CPU by default so this also works on Windows. Change `device` if you have a configured GPU.


In [ ]:
fiction_categories = ["Fiction", "Nonfiction"]
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=-1,
)

## Check a small sample

Compare predictions with up to 300 mapped books per category, using a fixed sampling seed. These mapped categories are rough reference labels. This is an exploratory sanity check, not a held-out quality result.


In [ ]:
sample_results = []
for category in fiction_categories:
    category_books = books.loc[books["simple_categories"] == category]
    sample = category_books.sample(n=min(300, len(category_books)), random_state=42)

    for _, book in tqdm(sample.iterrows(), total=len(sample), desc=category):
        prediction = predict_category(
            book["description"], classifier, fiction_categories
        )
        sample_results.append({"actual": category, "predicted": prediction})

results = pd.DataFrame(sample_results, columns=["actual", "predicted"])
if results.empty:
    print("No mapped Fiction or Nonfiction books to check.")
else:
    results["correct"] = results["actual"] == results["predicted"]
    print(f"Agreement with mapped categories: {results['correct'].mean():.1%}")
results.head()

## Fill in missing categories

Keep existing category mappings, including the children's categories.


In [ ]:
missing_categories = books.loc[books["simple_categories"].isna()]

for index, book in tqdm(missing_categories.iterrows(), total=len(missing_categories)):
    prediction = predict_category(book["description"], classifier, fiction_categories)
    books.loc[index, "simple_categories"] = prediction

books["simple_categories"].value_counts()

## Save the categories


In [ ]:
books.to_csv("books_with_categories.csv", index=False)
print(f"Saved {len(books)} books to books_with_categories.csv")